# 02 · SAINT Baseline (Home Credit)

SAINT architecture trained on the Home Credit Default Risk dataset. Provides a non-FTT transformer baseline that the FTT variants in `00_ftt_architecture_variants.ipynb` can be compared against.

## 1. Setup

In [ ]:
# %pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/preprocessed_data_home_credit.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/home_credit")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

Loads the preprocessed parquet file and drops the leakage / low-signal columns identified during EDA. Numerical and categorical column lists are explicit to keep the pipeline reproducible.

In [ ]:
# Load preprocessed Home Credit data + drop the leakage / low-signal columns
# identified during initial EDA.
df = pd.read_parquet(DATA_PATH)

DROP_COLUMNS_HC = [
    "FLAG_PHONE", "OCCUPATION_TYPE", "SELLERPLACE_AREA", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_ATM_CURRENT", "CODE_REJECT_REASON_HC", "AMT_PAYMENT_TOTAL_CURRENT",
    "CHANNEL_TYPE_AP+ (Cash loan)", "NUM_INSTALMENT_NUMBER_x", "CNT_INSTALMENT_MATURE_CUM",
    "CREDIT_TYPE_Car loan", "WEEKDAY_APPR_PROCESS_START_MONDAY",
    "CHANNEL_TYPE_Credit and cash offices", "NAME_YIELD_GROUP_middle",
    "AMT_DRAWINGS_CURRENT", "SK_ID_PREV_y", "CHANNEL_TYPE_Channel of corporate sales",
    "SK_DPD_y", "WEEKDAY_APPR_PROCESS_START_WEDNESDAY", "MONTHS_MAX",
    "NAME_GOODS_CATEGORY_Clothing and Accessories", "CODE_REJECT_REASON_SCO",
    "OBS_30_CNT_SOCIAL_CIRCLE", "AMT_REQ_CREDIT_BUREAU_YEAR",
    "PRODUCT_COMBINATION_Cash Street: middle",
    "NAME_GOODS_CATEGORY_Photo / Cinema Equipment",
    "PRODUCT_COMBINATION_Cash X-Sell: middle", "AMT_APPLICATION", "DAYS_FIRST_DUE",
    "FLAG_DOCUMENT_16", "AMT_ANNUITY", "NAME_SELLER_INDUSTRY_Connectivity",
    "PRODUCT_COMBINATION_POS industry without interest", "NONLIVINGAPARTMENTS_AVG",
    "NAME_SELLER_INDUSTRY_Industry", "DAYS_CREDIT_UPDATE", "RATE_DOWN_PAYMENT",
    "ELEVATORS_AVG", "STATUS_X", "ENTRANCES_AVG",
    "WEEKDAY_APPR_PROCESS_START_SATURDAY", "MONTHS_BALANCE_y",
    "AMT_CREDIT_SUM_OVERDUE", "NAME_PORTFOLIO_POS", "SK_DPD_x", "FLAG_DOCUMENT_13",
    "NAME_TYPE_SUITE_Unaccompanied", "WEEKDAY_APPR_PROCESS_START_FRIDAY",
    "FLAG_DOCUMENT_6", "CHANNEL_TYPE_Stone", "SK_DPD_DEF_y",
    "NAME_PRODUCT_TYPE_x-sell", "NONLIVINGAREA_AVG", "FLAG_DOCUMENT_18",
    "nb_app", "NAME_HOUSING_TYPE", "NAME_PORTFOLIO_Cash",
    "WEEKDAY_APPR_PROCESS_START_SUNDAY", "WEEKDAY_APPR_PROCESS_START_THURSDAY",
    "NFLAG_INSURED_ON_APPROVAL", "MONTHS_COUNT", "YEARS_BUILD_MEDI",
    "DAYS_FIRST_DRAWING", "CODE_REJECT_REASON_LIMIT", "NAME_YIELD_GROUP_XNA",
    "NAME_GOODS_CATEGORY_Medical Supplies", "CREDIT_TYPE_Consumer credit",
    "PRODUCT_COMBINATION_POS household with interest",
    "PRODUCT_COMBINATION_POS mobile with interest", "AMT_DRAWINGS_POS_CURRENT",
    "NAME_CONTRACT_TYPE_Revolving loans", "FLOORSMAX_AVG", "WALLSMATERIAL_MODE",
    "CHANNEL_TYPE_Contact center", "CREDIT_TYPE_Credit card",
    "CODE_REJECT_REASON_SCOFR", "NAME_SELLER_INDUSTRY_XNA", "BASEMENTAREA_AVG",
    "LANDAREA_MEDI", "NAME_SELLER_INDUSTRY_Consumer electronics", "NAME_TYPE_SUITE",
    "FLAG_LAST_APPL_PER_CONTRACT_Y", "CNT_CHILDREN",
    "PRODUCT_COMBINATION_Cash Street: high", "NAME_GOODS_CATEGORY_Sport and Leisure",
    "NAME_TYPE_SUITE_Children", "NAME_SELLER_INDUSTRY_Construction",
    "NAME_GOODS_CATEGORY_Tourism", "NAME_CASH_LOAN_PURPOSE_Repairs",
    "FLAG_CONT_MOBILE", "WEEKDAY_APPR_PROCESS_START_TUESDAY",
    "AMT_REQ_CREDIT_BUREAU_DAY", "NAME_GOODS_CATEGORY_Mobile",
    "NAME_TYPE_SUITE_Spouse, partner", "REG_REGION_NOT_LIVE_REGION",
    "CREDIT_TYPE_Another type of loan", "COMMONAREA_MEDI", "FONDKAPREMONT_MODE",
    "NAME_GOODS_CATEGORY_Construction Materials", "NAME_PORTFOLIO_XNA",
    "NAME_TYPE_SUITE_Family", "NAME_PAYMENT_TYPE_Non-cash from your account",
    "NAME_CASH_LOAN_PURPOSE_Other", "CNT_FAM_MEMBERS", "CREDIT_ACTIVE_Sold",
    "PRODUCT_COMBINATION_Card Street", "PRODUCT_COMBINATION_Cash",
    "NAME_TYPE_SUITE_Other_B", "NAME_CONTRACT_TYPE_Cash loans", "FLOORSMIN_AVG",
    "CHANNEL_TYPE_Country-wide", "NUM_INSTALMENT_VERSION",
    "NAME_CONTRACT_STATUS_Canceled", "AMT_REQ_CREDIT_BUREAU_MON",
    "NAME_TYPE_SUITE_Other_A", "NAME_SELLER_INDUSTRY_MLM partners",
    "NAME_GOODS_CATEGORY_Gardening", "REG_CITY_NOT_WORK_CITY",
    "CNT_CREDIT_PROLONG", "NUM_INSTALMENT_NUMBER",
    "NAME_GOODS_CATEGORY_Auto Accessories", "STATUS_C", "NAME_GOODS_CATEGORY_Jewelry",
    "AMT_REQ_CREDIT_BUREAU_WEEK", "CHANNEL_TYPE_Car dealer", "HOUSETYPE_MODE",
    "FLAG_MOBIL", "FLAG_EMAIL", "FLAG_EMP_PHONE",
    "LIVE_REGION_NOT_WORK_REGION", "LIVE_CITY_NOT_WORK_CITY",
    "NAME_GOODS_CATEGORY_Fitness", "NAME_GOODS_CATEGORY_Education",
    "NAME_GOODS_CATEGORY_Direct Sales", "NAME_GOODS_CATEGORY_Consumer Electronics",
    "NAME_GOODS_CATEGORY_Computers", "NAME_GOODS_CATEGORY_Audio/Video",
    "NAME_GOODS_CATEGORY_Additional Service", "NAME_GOODS_CATEGORY_Animals",
    "NAME_TYPE_SUITE_Group of people", "NAME_CLIENT_TYPE_Refreshed",
    "NAME_CLIENT_TYPE_XNA", "AMT_DRAWINGS_OTHER_CURRENT",
    "CREDIT_TYPE_Unknown type of loan", "CODE_REJECT_REASON_SYSTEM",
    "CODE_REJECT_REASON_XNA", "CODE_REJECT_REASON_VERIF",
    "CREDIT_TYPE_Real estate loan", "NUNIQUE_STATUS_x", "NUNIQUE_STATUS_y",
    "NUNIQUE_STATUS2_y", "CNT_DRAWINGS_OTHER_CURRENT",
    "REG_REGION_NOT_WORK_REGION", "NAME_GOODS_CATEGORY_House Construction",
    "NAME_GOODS_CATEGORY_Homewares", "NAME_CASH_LOAN_PURPOSE_Buying a new car",
    "NAME_CASH_LOAN_PURPOSE_Buying a used car", "NAME_CASH_LOAN_PURPOSE_Car repairs",
    "NAME_CASH_LOAN_PURPOSE_Education",
    "NAME_CASH_LOAN_PURPOSE_Business development",
    "NAME_CASH_LOAN_PURPOSE_Buying a garage", "FLAG_LAST_APPL_PER_CONTRACT_N",
    "NAME_CASH_LOAN_PURPOSE_Building a house or an annex",
    "NFLAG_LAST_APPL_IN_DAY", "CHANNEL_TYPE_Regional / Local",
    "NAME_SELLER_INDUSTRY_Auto technology", "FLAG_DOCUMENT_17",
    "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21",
    "AMT_REQ_CREDIT_BUREAU_HOUR", "NAME_GOODS_CATEGORY_Other",
    "NAME_GOODS_CATEGORY_Office Appliances", "NAME_GOODS_CATEGORY_Insurance",
    "NAME_GOODS_CATEGORY_Medicine", "NAME_GOODS_CATEGORY_Weapon",
    "NAME_GOODS_CATEGORY_Vehicles", "NAME_CASH_LOAN_PURPOSE_Buying a holiday home / land",
    "NAME_CASH_LOAN_PURPOSE_Buying a home", "NAME_CASH_LOAN_PURPOSE_Hobby",
    "NAME_CASH_LOAN_PURPOSE_Gasification / water supply",
    "NAME_CASH_LOAN_PURPOSE_Furniture", "NAME_CASH_LOAN_PURPOSE_Everyday expenses",
    "NAME_CASH_LOAN_PURPOSE_Money for a third person",
    "NAME_CASH_LOAN_PURPOSE_Payments on other loans",
    "NAME_CASH_LOAN_PURPOSE_Medicine", "NAME_CASH_LOAN_PURPOSE_Journey",
    "NAME_CASH_LOAN_PURPOSE_Refusal to name the goal",
    "NAME_CASH_LOAN_PURPOSE_Purchase of electronic equipment",
    "NAME_CASH_LOAN_PURPOSE_Wedding / gift / holiday",
    "NAME_CASH_LOAN_PURPOSE_Urgent needs",
    "NAME_CONTRACT_STATUS_Unused offer",
    "NAME_PAYMENT_TYPE_Cashless from the account of the employer",
    "NAME_CONTRACT_TYPE_XNA",
    "PRODUCT_COMBINATION_POS household without interest",
    "FLAG_DOCUMENT_5", "FLAG_DOCUMENT_4", "FLAG_DOCUMENT_2",
    "EMERGENCYSTATE_MODE",
    "PRODUCT_COMBINATION_POS others without interest",
    "PRODUCT_COMBINATION_POS other with interest",
    "PRODUCT_COMBINATION_POS mobile without interest", "CREDIT_DAY_OVERDUE",
    "NAME_SELLER_INDUSTRY_Tourism", "NAME_SELLER_INDUSTRY_Jewelry",
    "PRODUCT_COMBINATION_Card X-Sell", "FLAG_DOCUMENT_15",
    "FLAG_DOCUMENT_7", "FLAG_DOCUMENT_8", "FLAG_DOCUMENT_9",
    "FLAG_DOCUMENT_10", "CREDIT_CURRENCY_currency 4",
    "CREDIT_CURRENCY_currency 3", "CREDIT_CURRENCY_currency 2",
    "CREDIT_ACTIVE_Bad debt", "FLAG_DOCUMENT_11", "FLAG_DOCUMENT_12",
    "FLAG_DOCUMENT_14", "CREDIT_TYPE_Cash loan (non-earmarked)",
    "CREDIT_TYPE_Loan for the purchase of equipment",
    "CREDIT_TYPE_Loan for purchase of shares (margin lending)",
    "CREDIT_TYPE_Loan for business development",
    "CREDIT_TYPE_Interbank credit",
    "CREDIT_TYPE_Loan for working capital replenishment",
    "CREDIT_TYPE_Mobile operator loan", "FLAG_OWN_REALTY", "FLAG_OWN_CAR",
]
df = df.drop(columns=DROP_COLUMNS_HC)

NUM_COLS_HC = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT_x", "AMT_ANNUITY_x", "AMT_GOODS_PRICE_x",
    "REGION_POPULATION_RELATIVE", "DAYS_BIRTH", "DAYS_EMPLOYED",
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "OWN_CAR_AGE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "HOUR_APPR_PROCESS_START_x", "REG_CITY_NOT_LIVE_CITY",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "APARTMENTS_AVG", "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BUILD_AVG",
    "COMMONAREA_AVG", "LANDAREA_AVG", "LIVINGAPARTMENTS_AVG", "LIVINGAREA_AVG",
    "APARTMENTS_MODE", "BASEMENTAREA_MODE", "YEARS_BEGINEXPLUATATION_MODE",
    "YEARS_BUILD_MODE", "COMMONAREA_MODE", "ELEVATORS_MODE", "ENTRANCES_MODE",
    "FLOORSMAX_MODE", "FLOORSMIN_MODE", "LANDAREA_MODE", "LIVINGAPARTMENTS_MODE",
    "LIVINGAREA_MODE", "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAREA_MODE",
    "APARTMENTS_MEDI", "BASEMENTAREA_MEDI", "YEARS_BEGINEXPLUATATION_MEDI",
    "ELEVATORS_MEDI", "ENTRANCES_MEDI", "FLOORSMAX_MEDI", "FLOORSMIN_MEDI",
    "LIVINGAPARTMENTS_MEDI", "LIVINGAREA_MEDI", "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAREA_MEDI", "TOTALAREA_MODE", "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE",
    "DAYS_LAST_PHONE_CHANGE", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_ANNUITY_y",
    "AMT_CREDIT_y", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE_y",
    "HOUR_APPR_PROCESS_START_y", "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED", "DAYS_DECISION", "CNT_PAYMENT",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION",
    "NAME_CONTRACT_TYPE_Consumer loans", "NAME_CASH_LOAN_PURPOSE_XAP",
    "NAME_CASH_LOAN_PURPOSE_XNA", "NAME_CONTRACT_STATUS_Approved",
    "NAME_CONTRACT_STATUS_Refused", "NAME_PAYMENT_TYPE_Cash through the bank",
    "NAME_PAYMENT_TYPE_XNA", "CODE_REJECT_REASON_CLIENT",
    "CODE_REJECT_REASON_XAP", "NAME_CLIENT_TYPE_New",
    "NAME_CLIENT_TYPE_Repeater", "NAME_GOODS_CATEGORY_Furniture",
    "NAME_GOODS_CATEGORY_XNA", "NAME_PORTFOLIO_Cards", "NAME_PORTFOLIO_Cars",
    "NAME_PRODUCT_TYPE_XNA", "NAME_PRODUCT_TYPE_walk-in",
    "NAME_SELLER_INDUSTRY_Clothing", "NAME_SELLER_INDUSTRY_Furniture",
    "NAME_YIELD_GROUP_high", "NAME_YIELD_GROUP_low_action",
    "NAME_YIELD_GROUP_low_normal", "PRODUCT_COMBINATION_Cash Street: low",
    "PRODUCT_COMBINATION_Cash X-Sell: high", "PRODUCT_COMBINATION_Cash X-Sell: low",
    "PRODUCT_COMBINATION_POS industry with interest", "DAYS_CREDIT",
    "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "AMT_CREDIT_MAX_OVERDUE",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT",
    "STATUS_0", "STATUS_1", "STATUS_2", "STATUS_3", "STATUS_4", "STATUS_5",
    "MONTHS_MIN", "CREDIT_ACTIVE_Active", "CREDIT_ACTIVE_Closed",
    "CREDIT_CURRENCY_currency 1", "CREDIT_TYPE_Microloan", "CREDIT_TYPE_Mortgage",
    "buro_count", "MONTHS_BALANCE_x", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
    "SK_DPD_DEF_x", "NUNIQUE_STATUS2_x", "AMT_BALANCE",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT", "NUM_INSTALMENT_VERSION_x",
    "DAYS_INSTALMENT_x", "DAYS_ENTRY_PAYMENT_x", "AMT_INSTALMENT_x",
    "AMT_PAYMENT_x", "SK_ID_PREV_x", "NUM_INSTALMENT_VERSION_y",
    "NUM_INSTALMENT_NUMBER_y", "DAYS_INSTALMENT_y", "DAYS_ENTRY_PAYMENT_y",
    "AMT_INSTALMENT_y", "AMT_PAYMENT_y", "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT",
]
CAT_COLS_HC = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "WEEKDAY_APPR_PROCESS_START", "ORGANIZATION_TYPE", "EMERGENCYSTATE_MODE",
]
num_cols = NUM_COLS_HC
cat_cols = CAT_COLS_HC

## 3. Preprocessing Pipeline

Splits into 64 / 16 / 20 train / valid / test, applies a high-missing-value filter, drops highly correlated numeric features (keeping the one most correlated with the target), then imputes, label-encodes and standard-scales — all fit on train only to prevent leakage.

In [ ]:
def apply_missing_value_filter(X_train, X_valid, X_test, threshold=0.50):
    """Drop columns whose missing rate in X_train exceeds `threshold`."""
    missing_pct = X_train.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
    print("--- 2. Missing Value Column Filter ---")
    print(f"Missing percentage threshold: {threshold:.1%}")
    print(f"Columns dropped ({len(cols_to_drop)}): {cols_to_drop if cols_to_drop else 'None'}")

    X_train_clean = X_train.drop(columns=cols_to_drop, errors="ignore")
    X_valid_clean = X_valid.drop(columns=cols_to_drop, errors="ignore")
    X_test_clean  = X_test.drop(columns=cols_to_drop, errors="ignore")

    all_cols = X_train_clean.columns.tolist()
    num_cols_clean = X_train_clean.select_dtypes(include=np.number).columns.tolist()
    cat_cols_clean = [c for c in all_cols if c not in num_cols_clean]
    print(f"Features remaining: {X_train_clean.shape[1]}")
    print("-" * 50)
    return X_train_clean, X_valid_clean, X_test_clean, num_cols_clean, cat_cols_clean


def apply_correlation_drop(X_train, y_train, X_valid, X_test, threshold=0.9):
    """Drop one feature from each numeric pair whose |correlation| > threshold,
    keeping the one with higher absolute correlation to the target."""
    print("--- 3. Correlation Drop (numeric only) ---")
    all_cols = X_train.columns.tolist()
    num_cols_start = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols_start = [c for c in all_cols if c not in num_cols_start]
    X_train_num = X_train[num_cols_start]

    print(f"Numeric features before drop: {len(num_cols_start)}. Threshold: {threshold}")
    corr_matrix = X_train_num.corr().abs()
    target_corr = X_train_num.apply(lambda x: x.corr(y_train)).abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop_num = set()
    for col in upper.columns:
        for row in upper.index:
            if upper.loc[row, col] > threshold:
                a, b = row, col
                if a in to_drop_num or b in to_drop_num:
                    continue
                if target_corr.get(a, 0) < target_corr.get(b, 0):
                    to_drop_num.add(a)
                else:
                    to_drop_num.add(b)

    features_to_drop = list(to_drop_num)
    X_train_clean = X_train.drop(columns=features_to_drop, errors="ignore")
    X_valid_clean = X_valid.drop(columns=features_to_drop, errors="ignore")
    X_test_clean  = X_test.drop(columns=features_to_drop, errors="ignore")
    print(f"Numeric features dropped: {len(features_to_drop)}")
    print("-" * 50)

    num_cols_clean = [c for c in num_cols_start if c not in features_to_drop]
    cat_cols_clean = cat_cols_start
    return X_train_clean, X_valid_clean, X_test_clean, num_cols_clean, cat_cols_clean


def preprocess_data_pipeline(df, target="TARGET", corr_threshold=0.9, missing_threshold=0.50):
    """Full HC preprocessing pipeline: split, filter missing, drop correlated,
    impute, label-encode, scale. Returns processed splits and helper objects."""
    print("--- 1. Splitting Data (64/16/20) ---")
    X = df.drop(columns=[target])
    y = df[target]
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42,
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=42,
    )
    print(f"Split sizes: Train={len(X_train)} | Valid={len(X_valid)} | Test={len(X_test)}")
    print("-" * 50)

    num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

    X_train, X_valid, X_test, num_cols, cat_cols = apply_missing_value_filter(
        X_train, X_valid, X_test, threshold=missing_threshold,
    )
    X_train, X_valid, X_test, num_cols, cat_cols = apply_correlation_drop(
        X_train, y_train, X_valid, X_test, threshold=corr_threshold,
    )

    print("--- 4. Leakage-Free Transformations ---")
    median_imputers = {}
    if num_cols:
        print(f"Imputing {len(num_cols)} numeric columns with median...")
        for col in num_cols:
            median_value = X_train[col].median()
            median_imputers[col] = median_value
            X_train[col] = X_train[col].fillna(median_value)
            X_valid[col] = X_valid[col].fillna(median_value)
            X_test[col]  = X_test[col].fillna(median_value)

    encoders = {}
    cat_cardinalities = []
    for col in cat_cols:
        X_train[col] = X_train[col].fillna("Missing")
        X_valid[col] = X_valid[col].fillna("Missing")
        X_test[col]  = X_test[col].fillna("Missing")

        le = LabelEncoder()
        le.fit(X_train[col])
        new_classes = list(le.classes_)
        if "Missing" not in new_classes:
            new_classes.append("Missing")
        new_classes.append("UNKNOWN")
        le.classes_ = np.array(new_classes)
        X_train[col] = le.transform(X_train[col])
        encoders[col] = le
        cat_cardinalities.append(len(le.classes_))

        mapping = {cls: i for i, cls in enumerate(le.classes_)}
        unknown_id = mapping["UNKNOWN"]
        X_valid[col] = (X_valid[col].fillna("Missing").astype(str)
                                       .map(mapping).fillna(unknown_id).astype(int))
        X_test[col]  = (X_test[col].fillna("Missing").astype(str)
                                       .map(mapping).fillna(unknown_id).astype(int))

    print(f"Categorical features encoded: {len(cat_cols)}")

    scaler = StandardScaler()
    if num_cols:
        print(f"Scaling {len(num_cols)} numeric columns...")
        X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
        X_valid[num_cols] = scaler.transform(X_valid[num_cols])
        X_test[num_cols]  = scaler.transform(X_test[num_cols])
    print("-" * 50)

    print("--- Final Feature Summary ---")
    print(f"Numeric features      : {len(num_cols)}")
    print(f"Categorical features  : {len(cat_cols)}")
    print(f"Cat cardinalities     : {cat_cardinalities}")
    print(f"Total features        : {X_train.shape[1]}")
    print("-" * 50)

    return (
        X_train, y_train, X_valid, y_valid, X_test, y_test,
        num_cols, cat_cols, encoders, median_imputers, scaler, cat_cardinalities,
    )


def home_credit_summary(df, num_cols, cat_cols, target="TARGET"):
    print("\n" + "=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total Rows: {df.shape[0]:,}")
    print(f"Total Columns: {df.shape[1]:,}\n")
    print("Feature Breakdown")
    print(f"  Numeric Features     : {len(num_cols)}")
    print(f"  Categorical Features : {len(cat_cols)}")
    print(f"  Binary FLAG Columns  : {len([c for c in df.columns if 'FLAG' in c])}")
    print("-" * 50)
    if target in df.columns:
        target_dist = df[target].value_counts(normalize=True) * 100
        print("Target Distribution")
        print(target_dist.to_frame("Percentage %"))
        print("-" * 50)
    print("Missing Value Summary (Top 15)")
    missing = df.isnull().mean().sort_values(ascending=False)
    print((missing * 100).head(15).to_frame("Missing %"))
    print("=" * 60)


home_credit_summary(df, num_cols, cat_cols)
(
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    final_num_cols, final_cat_cols, encoders, imputers, scaler, cat_cardinalities_final,
) = preprocess_data_pipeline(df, target="TARGET", corr_threshold=0.9, missing_threshold=0.9)

print("\n================ Dataset Split Summary ================\n")
print(f"Train : X = {X_train.shape},  y = {y_train.shape}")
print(f"Valid : X = {X_valid.shape},  y = {y_valid.shape}")
print(f"Test  : X = {X_test.shape},   y = {y_test.shape}")
print(f"\nNumeric features      : {len(final_num_cols)}")
print(f"Categorical features  : {len(final_cat_cols)}")
print(f"Cat cardinalities     : {cat_cardinalities_final}")

In [ ]:
# Build torch tensors for FTT-style models.
data_numpy = {"train": {}, "val": {}, "test": {}}
data_numpy["train"]["x_cat"]  = X_train[final_cat_cols]
data_numpy["val"]["x_cat"]    = X_valid[final_cat_cols]
data_numpy["test"]["x_cat"]   = X_test[final_cat_cols]
data_numpy["train"]["X_cont"] = X_train[final_num_cols]
data_numpy["val"]["X_cont"]   = X_valid[final_num_cols]
data_numpy["test"]["X_cont"]  = X_test[final_num_cols]
data_numpy["train"]["y"]      = y_train
data_numpy["val"]["y"]        = y_valid
data_numpy["test"]["y"]       = y_test

data = {}
for part in data_numpy:
    data[part] = {
        "X_cont": torch.tensor(data_numpy[part]["X_cont"].to_numpy(), dtype=torch.float32, device=device),
        "x_cat":  torch.tensor(data_numpy[part]["x_cat"].to_numpy(),  dtype=torch.long,    device=device),
        "y":      torch.tensor(data_numpy[part]["y"].to_numpy(),      dtype=torch.float32, device=device),
    }

d_numerical = len(final_num_cols)
n_train = data["train"]["y"].shape[0]
print(f"d_numerical = {d_numerical}, n_train = {n_train}")

In [ ]:
# When DEV_MODE is on, subsample data to a few thousand rows so a full training
# pass completes in a couple of minutes. No-op when DEV_MODE is False.
if DEV_MODE:
    _n_dev = 5000
    for _part in data:
        _idx = torch.randperm(data[_part]["y"].shape[0], device=device)[:_n_dev]
        for _k in data[_part]:
            data[_part][_k] = data[_part][_k][_idx]
    print({_part: {k: v.shape for k, v in data[_part].items()} for _part in data})

## 4. Model Definition

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SaintAttention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.to_qkv = nn.Linear(dim, dim * 3, bias=False)
        self.to_out = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, n, d = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: t.view(b, n, self.heads, -1).transpose(1, 2), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = F.softmax(dots, dim=-1)
        out = torch.matmul(self.dropout(attn), v)
        out = out.transpose(1, 2).reshape(b, n, d)
        return self.to_out(out)

class SAINT(nn.Module):
    def __init__(self, categories, num_continuous, dim=32, depth=6, heads=8):
        super().__init__()
        self.num_categories = len(categories)
        self.num_continuous = num_continuous

        # Embeddings
        self.cat_embeds = nn.ModuleList([nn.Embedding(c, dim) for c in categories])
        self.cont_embed = nn.Linear(1, dim)

        # Transformer Layers
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                SaintAttention(dim, heads), # Column Attention
                nn.LayerNorm(dim),
                nn.Linear(dim, dim), # Feed Forward
                nn.LayerNorm(dim)
            ]))

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim * (self.num_categories + num_continuous)),
            nn.Linear(dim * (self.num_categories + num_continuous), 1)
        )

    def forward(self, x_cat, x_cont):
        # x_cat: [Batch, n_cat], x_cont: [Batch, n_cont]
        embeddings = []
        for i, emb in enumerate(self.cat_embeds):
            embeddings.append(emb(x_cat[:, i]))

        for i in range(self.num_continuous):
            embeddings.append(self.cont_embed(x_cont[:, i:i+1]))

        x = torch.stack(embeddings, dim=1) # [Batch, n_features, dim]

        for attn, norm1, ff, norm2 in self.layers:
            x = attn(x) + x
            x = norm1(x)
            x = ff(x) + x
            x = norm2(x)

        return self.mlp_head(x.flatten(1))

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

from torch.utils.data import DataLoader, TensorDataset

# # Define batch size here - remember, larger batches give SAINT more context!
# BATCH_SIZE = 1024

train_ds = TensorDataset(data["train"]["x_cat"], data["train"]["X_cont"], data["train"]["y"])
# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TensorDataset(data["val"]["x_cat"], data["val"]["X_cont"], data["val"]["y"])
# val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

test_ds = TensorDataset(data["test"]["x_cat"], data["test"]["X_cont"], data["test"]["y"])
# test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


# print(f"Created train_loader with {len(train_loader)} batches")
# print(f"Created val_loader with {len(val_loader)} batches")
# print(f"Created val_loader with {len(test_loader)} batches")

In [ ]:
def get_saint_batch_size(
    d_model: int,
    n_blocks: int,
    d_numerical: int,
    cat_cardinalities: list,
    n_heads: int,
    target_gpu_gb: float = 15.0,
    safety_factor: float = 0.50, # more conservative — SAINT has 2x attention
) -> int:
    """
    Dynamic batch size estimator for SAINT on single GPU (Colab T4/A100).

    SAINT has two attention mechanisms per block vs FTT's one:
    1. Column-wise (intersample) attention — same as FTT
    2. Row-wise (intersample) attention — attends across batch samples
       This makes row-wise attention O(B^2) in memory, not O(n_cols^2)
       So batch size appears in attention map size — critical difference from FTT.
    """
    total_cols = d_numerical + len(cat_cardinalities)
    B_bytes = 4 # bytes per float32

    # ── Model weights (static) ────────────────────────────────────────────────
    # Embeddings
    num_embed_params = d_numerical * d_model * 2 # weight + bias
    cat_embed_params = sum(cat_cardinalities) * d_model

    # Per block — SAINT has TWO attention sublayers per block
    # 1. Column attention (same as FTT SPA)
    col_attn_params = 4 * (d_model * d_model) # Q, K, V, O projections

    # 2. Row attention (intersample) — same param count as col attention
    row_attn_params = 4 * (d_model * d_model)

    # FFN after each attention sublayer — SAINT has 2 FFNs per block
    ffn_params_each = 2 * (d_model * d_model * 4) + (d_model * 4 * d_model)
    ffn_params_total = ffn_params_each * 2 # one per attention sublayer

    # LayerNorm — 4 per block (before each attention + before each FFN)
    ln_params = 4 * d_model * 2

    block_params = (
        col_attn_params + row_attn_params +
        ffn_params_total + ln_params
    ) * n_blocks

    # Decoder
    decoder_params = d_model + d_model * 1

    total_params = num_embed_params + cat_embed_params + block_params + decoder_params
    model_weights_mem = total_params * B_bytes

    # ── Optimizer states (Adam: m + v) ────────────────────────────────────────
    optimizer_mem = model_weights_mem * 2

    # ── Activations per sample (forward pass) ────────────────────────────────
    # Embedding activations
    embed_act = total_cols * d_model * B_bytes

    # ── Column attention activations (O(n_cols^2) — same as FTT) ─────────────
    col_qkv_act = 3 * total_cols * d_model * B_bytes
    col_attn_map = total_cols * total_cols * B_bytes # (n_cols, n_cols) per sample
    col_ffn_act = total_cols * d_model * 4 * B_bytes * 2
    col_block_act = col_qkv_act + col_attn_map + col_ffn_act

    # ── Row attention activations (O(B^2) — SAINT-specific, batch-dependent) ──
    # Attention map is (B, B) per head — this scales with batch size
    # We estimate per-sample cost as: B * n_heads * d_head activations
    # plus the full (B, B) attention map amortized per sample = B * B_bytes
    d_head = d_model // n_heads
    row_qkv_act = 3 * d_model * B_bytes # per sample Q,K,V
    # Amortized per-sample cost of (B x B) attention map
    # Will be multiplied by batch_size externally — treat as linear in B
    # Here we estimate at a reference batch of 1024 and solve below
    row_ffn_act = total_cols * d_model * 4 * B_bytes * 2
    row_block_act_base = row_qkv_act + row_ffn_act # excludes B-dependent map

    block_act_per_sample = (col_block_act + row_block_act_base) * n_blocks
    decoder_act = total_cols * d_model * B_bytes
    forward_base = embed_act + block_act_per_sample + decoder_act

    # ── Backward ≈ forward ────────────────────────────────────────────────────
    backward_base = forward_base
    base_per_sample = forward_base + backward_base

    # ── Row attention map scales as O(B^2) — solve for B ─────────────────────
    # Total memory = fixed + B * base_per_sample + B^2 * row_map_cost
    # row_map_cost per element = n_blocks * n_heads * B_bytes (one map per head per block)
    row_map_cost_per_pair = n_blocks * n_heads * B_bytes # bytes per (i,j) pair

    # ── CUDA overhead ─────────────────────────────────────────────────────────
    cuda_overhead_bytes = 2.0 * (1024 ** 3)

    budget_bytes = (
        target_gpu_gb * (1024 ** 3) * safety_factor
        - cuda_overhead_bytes
        - model_weights_mem
        - optimizer_mem
    )

    if budget_bytes <= 0:
        return 64

    # Solve quadratic: row_map_cost * B^2 + base_per_sample * B - budget = 0
    # B = (-base + sqrt(base^2 + 4 * row_map * budget)) / (2 * row_map)
    a = row_map_cost_per_pair
    b = base_per_sample
    c = -budget_bytes

    discriminant = b ** 2 - 4 * a * c
    if discriminant < 0:
        return 64

    raw = int((-b + discriminant ** 0.5) / (2 * a))
    batch_size = max(64, 1 << (raw.bit_length() - 1)) # round down to power of 2

    # ── Diagnostic prints ─────────────────────────────────────────────────────
    print(f"[SAINTBatchSizer] total_cols={total_cols} d_model={d_model} "
          f"n_blocks={n_blocks} n_heads={n_heads}")
    print(f" model={model_weights_mem/1e6:.1f}MB "
          f"optim={optimizer_mem/1e6:.1f}MB "
          f"cuda_overhead=2000MB")
    print(f" base_per_sample={base_per_sample/1e3:.2f}KB "
          f"row_map_cost_per_pair={row_map_cost_per_pair}B")
    print(f" budget={budget_bytes/1e9:.2f}GB "
          f"raw={raw} BATCH_SIZE={batch_size}")
    print(f" NOTE: row attention map at this batch = "
          f"{row_map_cost_per_pair * batch_size**2 / 1e6:.1f}MB")

    return batch_size

## 5. Optuna Optimization

In [ ]:
import json
import math
import datetime
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# ─────────────────────────────────────────────
# GLOBALS
# ─────────────────────────────────────────────
trial_results = []
global_best_auc = 0.0
global_model_path = str(MODELS_DIR / "saint_hc_best.pth")
RESULTS_FILE = str(RESULTS_DIR / "saint_hc_trials.json")

# ─────────────────────────────────────────────
# EVALUATE HELPER
# ─────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    for batch_cat, batch_cont, batch_y in loader:
        logits = model(
            batch_cat.to(device),
            batch_cont.to(device)
        ).squeeze(-1)
        probs = torch.sigmoid(logits)
        all_preds.extend(probs.cpu().numpy())
        all_targets.extend(batch_y.cpu().numpy())
    return roc_auc_score(all_targets, all_preds)

# ─────────────────────────────────────────────
# OBJECTIVE
# ─────────────────────────────────────────────
def objective(trial):
    global global_best_auc

    # ── Search Space ──────────────────────────
    dim = trial.suggest_categorical("dim", [32, 64, 128, 256])
    depth = trial.suggest_int("depth", 2, 8)
    heads = trial.suggest_categorical("heads", [2, 4, 8])
    lr = 1e-4
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    n_epochs = 50
    patience = 10

    # ── Dynamic batch size for single Colab GPU ──
    BATCH_SIZE = get_saint_batch_size(
        d_model=dim,
        n_blocks=depth,
        d_numerical=d_numerical,
        cat_cardinalities=cat_cardinalities_final,
        n_heads=heads,
        target_gpu_gb=15.0,
        safety_factor=0.50
    )

    # ── Print Trial Info ──────────────────────
    print("\n" + "=" * 70)
    print(f" Starting Trial {trial.number}")
    print(f" Started : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(" Hyperparameters:")
    print(f" dim = {dim}")
    print(f" depth = {depth}")
    print(f" heads = {heads}")
    print(f" lr = {lr:.2e}")
    print(f" weight_decay = {weight_decay:.2e}")
    print(f" batch_size = {BATCH_SIZE}")
    print(f" n_epochs = {n_epochs}")
    print(f" patience = {patience}")
    print("=" * 70 + "\n")

    # ── Rebuild loaders with dynamic batch size ──
    train_loader_trial = torch.utils.data.DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    val_loader_trial = torch.utils.data.DataLoader(
        val_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False
    )
    test_loader_trial = torch.utils.data.DataLoader(
        test_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False
    )

    # ── Model ─────────────────────────────────
    model = SAINT(
        categories=cat_cardinalities_final,
        num_continuous=d_numerical,
        dim=dim,
        depth=depth,
        heads=heads,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f" Trainable params: {n_params:,}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr * 10, # updated from 5 per earlier findings
        steps_per_epoch=len(train_loader_trial),
        epochs=n_epochs,
        pct_start=0.1, # updated from 0.3 per earlier findings
        anneal_strategy="cos",
    )
    warmup_epochs = max(1, int(n_epochs * 0.1)) # matches pct_start=0.1
    criterion = nn.BCEWithLogitsLoss()

    # ── Early Stopping State ──────────────────
    best = {"val": -math.inf, "test": -math.inf, "epoch": -1}
    epochs_no_improve = 0
    timer_start = datetime.datetime.now()

    print(f"Score before training: Val AUC = {evaluate(model, val_loader_trial, device):.4f}")
    print(f"Device: {device}")
    print("-" * 70)


    # ── Training Loop ─────────────────────────
    for epoch in range(n_epochs):
        model.train()
        trial_pruned=False
        train_loss = 0.0
        n_batches = 0

        pbar = tqdm(
            train_loader_trial,
            desc=f"Epoch {epoch+1}/{n_epochs} [Train]"
        )
        for batch_cat, batch_cont, batch_y in pbar:
            optimizer.zero_grad()
            logits = model(
                batch_cat.to(device),
                batch_cont.to(device)
            ).squeeze(-1)
            loss = criterion(logits, batch_y.to(device).float())
            loss.backward()
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = train_loss / max(n_batches, 1)

        # ── Validation ────────────────────────
        val_auc = evaluate(model, val_loader_trial, device)
        test_auc = evaluate(model, test_loader_trial, device)
        train_auc = evaluate(model, train_loader_trial, device)
        elapsed = str(datetime.datetime.now() - timer_start).split(".")[0]

        print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} "
              f"Test {test_auc:.4f} Loss {avg_train_loss:.4f} [{elapsed}]")

        # ── Early stopping + best tracking ────
        if val_auc > best["val"]:
            print(" New best epoch! ")
            best = {"val": val_auc, "test": test_auc, "epoch": epoch + 1}
            epochs_no_improve = 0
            if val_auc > global_best_auc:
                torch.save(model.state_dict(), global_model_path)
                global_best_auc = val_auc
                print(f" Saved new global best: {global_best_auc:.4f}")
        else:
            epochs_no_improve += 1
            if epoch >= warmup_epochs and epochs_no_improve >= patience:
                print(f"⏹ Early stopping at epoch {epoch + 1}")
                break

        # ── Optuna pruning ────────────────────
        trial.report(val_auc, epoch)
        if epoch >= warmup_epochs and trial.should_prune():
          print("️ Trial pruned by Optuna")
          trial_pruned=True
          break

    total_time = str(datetime.datetime.now() - timer_start).split(".")[0]

    # ── Save Trial Result ─────────────────────
    trial_record = {
        "run_id": datetime.datetime.now().strftime("%Y%m%d_%H%M%S"),
        "trial_number": trial.number,
        "dataset": "home_credit",
        "model": "SAINT",
        "model_config": {
            "dim": dim,
            "depth": depth,
            "heads": heads,
            "d_numerical": d_numerical,
            "n_cat_features": len(cat_cardinalities_final),
            "n_params": n_params,
        },
        "optimizer_config": {
            "optimizer": "AdamW",
            "lr": lr,
            "weight_decay": weight_decay,
            "scheduler": "OneCycleLR",
            "max_lr_multiplier": 10,
            "pct_start": 0.1,
            "batch_size": BATCH_SIZE,
        },
        "training": {
            "n_epochs": n_epochs,
            "patience": patience,
            "stopped_epoch": best["epoch"],
            "total_time": total_time,
        },
        "results": {
            "auc_val": best["val"],
            "auc_test": best["test"],
            "auc_train": train_auc,
            "final_epoch_val": val_auc,
            "final_epoch_test": test_auc,
        },
        "best": best,
    }
    trial_results.append(trial_record)

    with open(RESULTS_FILE, "w") as f:
        json.dump(trial_results, f, indent=4)
    print(f"\nTrial {trial.number} complete — best: {best}")
    print(f"Results saved to {RESULTS_FILE}")

    if trial_pruned:
      raise optuna.exceptions.TrialPruned()

    return best["val"]

# ─────────────────────────────────────────────
# RUN STUDY
# ─────────────────────────────────────────────
storage_name = "sqlite:///saint_hyperparam_search_home.db"
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=3
)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    storage=storage_name,
    load_if_exists=True,
    study_name="saint_hyperparam_search_home",
)
study.optimize(objective, n_trials=(1 if DEV_MODE else 30))